In [2]:
from google.colab import files

uploaded = files.upload()

Saving gprofiler_down.csv to gprofiler_down.csv
Saving gprofiler_up.csv to gprofiler_up.csv
Saving expression_log2_normalized.csv to expression_log2_normalized.csv
Saving protein_annotation.csv to protein_annotation.csv
Saving sample_annotation.csv to sample_annotation.csv


In [4]:
# ============================================================
# PR1 PROTEOMICS
# FIVE PATHWAY-BASED HEATMAPS
#
# Comparison: AS vs CON
# Sex-stratified groups:
# CON-F | CON-M | AS-F | AS-M
#
# Uses:
# - Observed log2-normalized expression values
# - gProfiler enriched gene sets
# - Protein annotation
# - Peptide-support filtering
#
# Outputs:
# - One PNG per heatmap
# - Protein membership CSV per category
# - Enrichment terms CSV per category
# - Summary CSV
# - One ZIP with all output files
# ============================================================


# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

import os
import re
import zipfile
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

sns.set_theme(
    style="white",
    font="DejaVu Sans"
)


# ------------------------------------------------------------
# 2. USER SETTINGS
# ------------------------------------------------------------

EXPRESSION_FILE = "expression_log2_normalized.csv"
PROTEIN_ANNOTATION_FILE = "protein_annotation.csv"
SAMPLE_ANNOTATION_FILE = "sample_annotation.csv"

GPROFILER_UP_FILE = "gprofiler_up.csv"
GPROFILER_DOWN_FILE = "gprofiler_down.csv"

OUTPUT_DIR = "heatmap_outputs"

# At least 50% of samples in EACH subgroup
# must have an observed protein intensity.

MIN_VALID_FRAC_PER_GROUP = 0.50

# Save PDF versions as well?
# False = one PNG per category
# True = one PNG and one PDF per category

SAVE_PDF = False

# Fixed order of the four groups

GROUP_ORDER = [
    "CON_F",
    "CON_M",
    "AS_F",
    "AS_M"
]

DISPLAY_GROUPS = {
    "CON_F": "CON-F",
    "CON_M": "CON-M",
    "AS_F": "AS-F",
    "AS_M": "AS-M"
}


# ------------------------------------------------------------
# 3. CHECK THAT ALL FILES ARE AVAILABLE
# ------------------------------------------------------------

required_files = [
    EXPRESSION_FILE,
    PROTEIN_ANNOTATION_FILE,
    SAMPLE_ANNOTATION_FILE,
    GPROFILER_UP_FILE,
    GPROFILER_DOWN_FILE
]

missing_files = [
    f for f in required_files
    if not Path(f).exists()
]

if missing_files:

    raise FileNotFoundError(
        "Missing files:\n"
        + "\n".join(missing_files)
        + "\nPlease upload them using Cell 1."
    )


# ------------------------------------------------------------
# 4. LOAD DATA
# ------------------------------------------------------------

expression_raw = pd.read_csv(
    EXPRESSION_FILE
)

protein_annotation_raw = pd.read_csv(
    PROTEIN_ANNOTATION_FILE
)

sample_annotation = pd.read_csv(
    SAMPLE_ANNOTATION_FILE
)

gprofiler_up = pd.read_csv(
    GPROFILER_UP_FILE
)

gprofiler_down = pd.read_csv(
    GPROFILER_DOWN_FILE
)


print("FILES LOADED")
print("------------------------------")

print("Expression:", expression_raw.shape)
print("Protein annotation:", protein_annotation_raw.shape)
print("Sample annotation:", sample_annotation.shape)
print("gProfiler UP:", gprofiler_up.shape)
print("gProfiler DOWN:", gprofiler_down.shape)


# ------------------------------------------------------------
# 5. CHECK REQUIRED COLUMNS
# ------------------------------------------------------------

def check_columns(df, required_columns, file_name):

    missing = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing:

        raise ValueError(
            f"Missing columns in {file_name}: {missing}"
        )


check_columns(
    expression_raw,
    ["protein_group_id"],
    EXPRESSION_FILE
)

check_columns(
    protein_annotation_raw,
    [
        "protein_group_id",
        "gene_name",
        "low_peptide_support"
    ],
    PROTEIN_ANNOTATION_FILE
)

check_columns(
    sample_annotation,
    [
        "sample",
        "sex"
    ],
    SAMPLE_ANNOTATION_FILE
)

for file_name, df in [
    (GPROFILER_UP_FILE, gprofiler_up),
    (GPROFILER_DOWN_FILE, gprofiler_down)
]:

    check_columns(
        df,
        ["name", "genes"],
        file_name
    )


# ------------------------------------------------------------
# 6. PREPARE EXPRESSION MATRIX
# ------------------------------------------------------------

expression_raw["protein_group_id"] = (
    expression_raw["protein_group_id"]
    .astype(str)
    .str.strip()
)

if expression_raw["protein_group_id"].duplicated().any():

    raise ValueError(
        "Duplicate protein_group_id values found."
    )


expression = (
    expression_raw
    .set_index("protein_group_id")
    .copy()
)

expression = expression.apply(
    pd.to_numeric,
    errors="coerce"
)


print("\nExpression matrix:", expression.shape)


# ------------------------------------------------------------
# 7. PREPARE SAMPLE ANNOTATION
# ------------------------------------------------------------

sample_annotation["sample"] = (
    sample_annotation["sample"]
    .astype(str)
    .str.strip()
)

if sample_annotation["sample"].duplicated().any():

    raise ValueError(
        "Duplicate sample IDs found in sample annotation."
    )


# Derive condition from sample ID:
# CON_... = controls
# AS_...  = aortic stenosis
# MR_...  = mitral regurgitation

sample_annotation["condition"] = (
    sample_annotation["sample"]
    .str.extract(r"^(CON|AS|MR)_", expand=False)
)


if sample_annotation["condition"].isna().any():

    raise ValueError(
        "Could not identify condition for some sample IDs."
    )


# Convert sex to F / M

sex_map = {
    "female": "F",
    "male": "M",
    "f": "F",
    "m": "M"
}

sample_annotation["sex_code"] = (
    sample_annotation["sex"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(sex_map)
)


# Fallback: infer sex from the sample ID

sex_from_id = (
    sample_annotation["sample"]
    .str.extract(r"_([FM])$", expand=False)
    .str.upper()
)

sample_annotation["sex_code"] = (
    sample_annotation["sex_code"]
    .fillna(sex_from_id)
)


if sample_annotation["sex_code"].isna().any():

    raise ValueError(
        "Could not identify sex for some samples."
    )


sample_annotation["plot_group"] = (
    sample_annotation["condition"]
    + "_"
    + sample_annotation["sex_code"]
)


# Verify sample IDs

expression_samples = set(expression.columns)

annotation_samples = set(
    sample_annotation["sample"]
)

missing_in_expression = (
    annotation_samples - expression_samples
)

extra_in_expression = (
    expression_samples - annotation_samples
)

if missing_in_expression or extra_in_expression:

    raise ValueError(
        "Sample ID mismatch.\n"
        f"Missing in expression: {sorted(missing_in_expression)}\n"
        f"Extra in expression: {sorted(extra_in_expression)}"
    )


# ------------------------------------------------------------
# 8. DEFINE THE FOUR GROUPS
# ------------------------------------------------------------

group_samples = {}

for group in GROUP_ORDER:

    samples = sample_annotation.loc[
        sample_annotation["plot_group"] == group,
        "sample"
    ].tolist()

    if len(samples) == 0:

        raise ValueError(
            f"No samples found for {group}."
        )

    group_samples[group] = samples


print("\nSAMPLE COUNTS")
print("------------------------------")

for group in GROUP_ORDER:

    print(
        DISPLAY_GROUPS[group],
        ":",
        len(group_samples[group])
    )


# ------------------------------------------------------------
# 9. CALCULATE GROUP MEANS AND COVERAGE
# ------------------------------------------------------------

# Use observed values from expression_log2_normalized.csv.
#
# Do NOT use the imputed expression matrix here.
#
# Missing values remain NaN.
#
# For each protein, calculate:
# - Mean log2 intensity in each subgroup
# - Fraction of samples with observed intensity


group_means = pd.DataFrame(
    index=expression.index
)

group_coverage = pd.DataFrame(
    index=expression.index
)


for group in GROUP_ORDER:

    samples = group_samples[group]

    group_means[group] = (
        expression[samples]
        .mean(axis=1, skipna=True)
    )

    group_coverage[group] = (
        expression[samples]
        .notna()
        .sum(axis=1)
        / len(samples)
    )


# ------------------------------------------------------------
# 10. PREPARE PROTEIN ANNOTATION
# ------------------------------------------------------------

protein_annotation = protein_annotation_raw.copy()

protein_annotation["protein_group_id"] = (
    protein_annotation["protein_group_id"]
    .astype(str)
    .str.strip()
)


if protein_annotation["protein_group_id"].duplicated().any():

    raise ValueError(
        "Duplicate protein_group_id values in protein annotation."
    )


# Convert low_peptide_support robustly to Boolean

def parse_boolean(value):

    if pd.isna(value):
        return True  # conservatively exclude unknown support

    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    value = str(value).strip().lower()

    if value in ["true", "1", "yes", "y"]:
        return True

    if value in ["false", "0", "no", "n"]:
        return False

    raise ValueError(
        f"Unrecognized low_peptide_support value: {value}"
    )


protein_annotation["low_peptide_support_bool"] = (
    protein_annotation["low_peptide_support"]
    .apply(parse_boolean)
)


# Count low-support protein groups

n_low_support = int(
    protein_annotation["low_peptide_support_bool"].sum()
)


print("\nPEPTIDE SUPPORT FILTER")
print("------------------------------")

print(
    "Protein groups excluded due to low peptide support:",
    n_low_support
)


# Keep only protein groups with adequate peptide support

protein_annotation_filtered = (
    protein_annotation.loc[
        ~protein_annotation["low_peptide_support_bool"]
    ].copy()
)


# Keep only protein groups present in expression matrix

protein_annotation_filtered = (
    protein_annotation_filtered.loc[
        protein_annotation_filtered["protein_group_id"]
        .isin(expression.index)
    ].copy()
)


# ------------------------------------------------------------
# 11. HELPER FUNCTIONS FOR GENE SYMBOLS
# ------------------------------------------------------------

def parse_annotation_symbols(value):

    """
    Split gene_name entries such as:
        COL1A1
        TGM2;TGM5
        PLG;PLGLB1

    Returns uppercase gene symbols.
    """

    if pd.isna(value):
        return set()

    parts = re.split(
        r"[;,|]",
        str(value)
    )

    return {
        part.strip().upper()
        for part in parts
        if part.strip()
        and part.strip().lower() not in ["nan", "na"]
    }


def parse_gprofiler_genes(value):

    """
    Split comma-separated gProfiler gene lists.
    """

    if pd.isna(value):
        return set()

    parts = re.split(
        r"[,;]",
        str(value)
    )

    return {
        part.strip().upper()
        for part in parts
        if part.strip()
    }


protein_annotation_filtered["gene_symbols"] = (
    protein_annotation_filtered["gene_name"]
    .apply(parse_annotation_symbols)
)


# ------------------------------------------------------------
# 12. DEFINE BIOLOGICAL CATEGORIES
# ------------------------------------------------------------

# IMPORTANT:
#
# These categories are based on terms present in your
# uploaded gProfiler UP and DOWN files.
#
# Some genes may belong to more than one category because
# enriched GO / Reactome / KEGG terms overlap biologically.
#
# ECM manual curation:
# We remove the following from the ECM heatmap:
# - Complement / coagulation / fibrinolysis:
#       FGA, FGB, FGG, PLG, SERPINF2, C1QC
# - Cytoskeleton / adhesion:
#       ACTN1, PLEC
#
# These genes remain eligible for the relevant other
# category heatmaps where their enriched terms include them.


ECM_EXCLUDE_GENES = {
    "FGA",
    "FGB",
    "FGG",
    "PLG",
    "SERPINF2",
    "C1QC",
    "ACTN1",
    "PLEC"
}


CATEGORY_SPECS = [

    # --------------------------------------------------------
    # CATEGORY 1: ECM
    # --------------------------------------------------------

    {
        "slug": "01_ECM",
        "title": "ECM composition and remodeling",
        "source": "up",

        "terms": [

            "extracellular matrix",
            "external encapsulating structure",
            "Extracellular matrix organization",
            "collagen trimer",
            "Collagen chain trimerization",
            "ECM proteoglycans",
            "Degradation of the extracellular matrix",
            "collagen fibril organization"

        ],

        "exclude_genes": ECM_EXCLUDE_GENES
    },


    # --------------------------------------------------------
    # CATEGORY 2: MITOCHONDRIA
    # --------------------------------------------------------

    {
        "slug": "02_Mitochondria",
        "title": "Energy metabolism and mitochondria",
        "source": "down",

        "terms": [

            "mitochondrion",
            "mitochondrial matrix",
            "mitochondrial gene expression"

        ],

        "exclude_genes": set()
    },


    # --------------------------------------------------------
    # CATEGORY 3: PROTEIN SYNTHESIS
    # --------------------------------------------------------

    {
        "slug": "03_Protein_synthesis",
        "title": "Protein synthesis and ribosomal machinery",
        "source": "down",

        "terms": [

            "Translation",
            "Ribosome",
            "ribosomal subunit",
            "ribosome",
            "translation",
            "protein biosynthetic process",
            "SRP-dependent cotranslational protein targeting to membrane"

        ],

        "exclude_genes": set()
    },


    # --------------------------------------------------------
    # CATEGORY 4: CYTOSKELETON
    # --------------------------------------------------------

    {
        "slug": "04_Cytoskeleton",
        "title": "Cytoskeleton, adhesion and muscle contraction",
        "source": "up",

        "terms": [

            "Cytoskeleton in muscle cells",
            "cell adhesion",
            "Integrin cell surface interactions",
            "Integrin signaling",
            "Non-integrin membrane-ECM interactions"

        ],

        "exclude_genes": set()
    },


    # --------------------------------------------------------
    # CATEGORY 5: COMPLEMENT / COAGULATION
    # --------------------------------------------------------

    {
        "slug": "05_Complement_coagulation",
        "title": "Complement, coagulation and immune-associated proteins",
        "source": "up",

        "terms": [

            "blood microparticle",

            "Complement and coagulation cascades",
            "Complement cascade",
            "Regulation of Complement cascade",

            "complement activation",
            "complement activation, classical pathway",
            "complement activation, alternative pathway",
            "Initial triggering of complement",

            "humoral immune response",

            "regulation of coagulation",
            "negative regulation of coagulation",
            "positive regulation of coagulation",

            "regulation of blood coagulation",
            "negative regulation of blood coagulation",
            "positive regulation of blood coagulation",

            "coagulation",
            "blood coagulation",
            "blood coagulation, fibrin clot formation",

            "fibrinolysis",
            "regulation of fibrinolysis",
            "negative regulation of fibrinolysis",

            "regulation of hemostasis",
            "negative regulation of hemostasis",
            "positive regulation of hemostasis",

            "hemostasis",
            "Hemostasis",

            "Formation of Fibrin Clot (Clotting Cascade)",
            "Common Pathway of Fibrin Clot Formation",
            "Intrinsic Pathway of Fibrin Clot Formation",

            "Platelet degranulation",
            "Platelet activation, signaling and aggregation",
            "Response to elevated platelet cytosolic Ca2+",

            "platelet alpha granule",
            "platelet alpha granule lumen"

        ],

        "exclude_genes": set()
    }

]


# ------------------------------------------------------------
# 13. CREATE OUTPUT FOLDER
# ------------------------------------------------------------

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# 14. EXTRACT GENES FROM ENRICHED TERMS
# ------------------------------------------------------------

def extract_category_genes(category_spec):

    if category_spec["source"] == "up":

        source_df = gprofiler_up.copy()

    elif category_spec["source"] == "down":

        source_df = gprofiler_down.copy()

    else:

        raise ValueError("Unknown gProfiler source.")


    # Normalize term names:
    # - strip extra spaces
    # - compare case-insensitively

    source_df["_name_clean"] = (
        source_df["name"]
        .astype(str)
        .str.strip()
        .str.casefold()
    )


    requested_terms = {
        str(term).strip().casefold()
        for term in category_spec["terms"]
    }


    selected_terms = source_df.loc[
        source_df["_name_clean"].isin(requested_terms)
    ].copy()


    found_terms = set(
        selected_terms["_name_clean"]
    )


    missing_terms = sorted(
        requested_terms - found_terms
    )


    # Extract gene symbols from all selected terms

    selected_genes = set()

    for gene_list in selected_terms["genes"]:

        selected_genes.update(
            parse_gprofiler_genes(gene_list)
        )


    selected_terms = selected_terms.drop(
        columns=["_name_clean"]
    )


    return (
        selected_genes,
        selected_terms,
        missing_terms
    )


# ------------------------------------------------------------
# 15. MAP GENES TO PROTEIN GROUPS
# ------------------------------------------------------------

def map_genes_to_proteins(selected_genes):

    annotation = protein_annotation_filtered.copy()


    annotation["matching_gene_symbols"] = (
        annotation["gene_symbols"].apply(
            lambda symbols: sorted(
                symbols.intersection(selected_genes)
            )
        )
    )


    matched = annotation.loc[
        annotation["matching_gene_symbols"].apply(len) > 0
    ].copy()


    # Convert matched symbols to readable text

    matched["matching_gene_symbols_text"] = (
        matched["matching_gene_symbols"]
        .apply(lambda x: "; ".join(x))
    )


    matched_ids = (
        matched["protein_group_id"]
        .drop_duplicates()
        .tolist()
    )


    # Determine which selected genes are not represented
    # in the peptide-filtered protein annotation.

    annotated_gene_union = set()

    for symbols in annotation["gene_symbols"]:

        annotated_gene_union.update(symbols)


    unmatched_genes = (
        selected_genes - annotated_gene_union
    )


    return (
        matched,
        matched_ids,
        unmatched_genes
    )


# ------------------------------------------------------------
# 16. FUNCTION TO CREATE ONE CATEGORY HEATMAP
# ------------------------------------------------------------

def create_category_heatmap(category_spec):


    print("\n")
    print("=" * 70)
    print(category_spec["title"])
    print("=" * 70)


    # --------------------------------------------------------
    # A. Extract genes from enriched terms
    # --------------------------------------------------------

    selected_genes_original, selected_terms, missing_terms = (
        extract_category_genes(category_spec)
    )


    print(
        "Enriched terms found:",
        len(selected_terms)
    )

    print(
        "Gene symbols before manual curation:",
        len(selected_genes_original)
    )


    if missing_terms:

        print(
            "Terms not found in gProfiler file:",
            missing_terms
        )


    # --------------------------------------------------------
    # B. Apply manual category exclusions
    # --------------------------------------------------------

    excluded_genes = (
        selected_genes_original
        .intersection(category_spec["exclude_genes"])
    )


    selected_genes = (
        selected_genes_original - excluded_genes
    )


    print(
        "Genes reclassified out of this category:",
        sorted(excluded_genes)
    )

    print(
        "Gene symbols after curation:",
        len(selected_genes)
    )


    if len(selected_genes) == 0:

        print("No genes selected. Skipping.")

        return None


    # --------------------------------------------------------
    # C. Map gene symbols to protein groups
    # --------------------------------------------------------

    matched_annotation, matched_ids, unmatched_genes = (
        map_genes_to_proteins(selected_genes)
    )


    print(
        "Matched protein groups after peptide filtering:",
        len(matched_ids)
    )

    print(
        "Selected genes not matched to protein annotation:",
        sorted(unmatched_genes)
    )


    if len(matched_ids) == 0:

        print("No protein groups matched. Skipping.")

        return None


    # --------------------------------------------------------
    # D. Extract group means and coverage
    # --------------------------------------------------------

    means = group_means.loc[
        matched_ids,
        GROUP_ORDER
    ].copy()


    coverage = group_coverage.loc[
        matched_ids,
        GROUP_ORDER
    ].copy()


    # --------------------------------------------------------
    # E. Apply minimum coverage filter
    # --------------------------------------------------------

    # Each protein must have at least 50% observed values
    # in each of CON-F, CON-M, AS-F and AS-M.

    valid_coverage_mask = (
        coverage >= MIN_VALID_FRAC_PER_GROUP
    ).all(axis=1)


    means_valid = means.loc[
        valid_coverage_mask
    ].copy()


    coverage_valid = coverage.loc[
        valid_coverage_mask
    ].copy()


    print(
        "Protein groups passing coverage filter:",
        len(means_valid)
    )


    if len(means_valid) < 2:

        print(
            "Not enough proteins passed coverage filter."
        )

        return None


    # --------------------------------------------------------
    # F. Calculate row-wise Z-scores
    # --------------------------------------------------------

    # Center and scale each protein across the four groups.
    #
    # Z = (group mean - row mean) / row standard deviation
    #
    # ddof=1 uses sample standard deviation.

    row_mean = means_valid.mean(axis=1)

    row_sd = means_valid.std(
        axis=1,
        ddof=1
    )


    # Remove proteins with no variation across the four groups

    variable_mask = row_sd > 1e-8


    means_valid = means_valid.loc[
        variable_mask
    ]

    row_mean = row_mean.loc[
        variable_mask
    ]

    row_sd = row_sd.loc[
        variable_mask
    ]

    coverage_valid = coverage_valid.loc[
        variable_mask
    ]


    if len(means_valid) < 2:

        print(
            "Not enough variable proteins for clustering."
        )

        return None


    z_scores = means_valid.sub(
        row_mean,
        axis=0
    ).div(
        row_sd,
        axis=0
    )


    # --------------------------------------------------------
    # G. Create readable protein labels
    # --------------------------------------------------------

    annotation_lookup = (
        protein_annotation_filtered
        .set_index("protein_group_id")
    )


    labels = {}

    for protein_id in z_scores.index:

        gene_name = annotation_lookup.loc[
            protein_id,
            "gene_name"
        ]


        if pd.isna(gene_name) or str(gene_name).strip() == "":

            gene_name = protein_id


        labels[protein_id] = str(gene_name)


    # If two protein groups have the same gene label,
    # append the protein group ID to distinguish them.

    label_counts = pd.Series(
        list(labels.values())
    ).value_counts()


    for protein_id in labels:

        current_label = labels[protein_id]

        if label_counts[current_label] > 1:

            labels[protein_id] = (
                current_label
                + " ["
                + protein_id
                + "]"
            )


    z_plot = z_scores.copy()

    z_plot.index = [
        labels[protein_id]
        for protein_id in z_scores.index
    ]

    z_plot.columns = [
        DISPLAY_GROUPS[group]
        for group in GROUP_ORDER
    ]


    # --------------------------------------------------------
    # H. Determine figure dimensions
    # --------------------------------------------------------

    n_proteins = len(z_plot)


    figure_height = max(
        8,
        min(
            24,
            4 + 0.13 * n_proteins
        )
    )


    if n_proteins > 120:

        ytick_fontsize = 5

    elif n_proteins > 80:

        ytick_fontsize = 6

    else:

        ytick_fontsize = 7


    # --------------------------------------------------------
    # I. CREATE CLUSTERED HEATMAP
    # --------------------------------------------------------

    cluster = sns.clustermap(

        z_plot,

        # Cluster protein rows
        row_cluster=True,

        # Keep biological groups in fixed order
        col_cluster=False,

        # Color scale
        cmap="vlag",
        center=0,
        vmin=-2,
        vmax=2,

        # Figure size
        figsize=(12, figure_height),

        # Labels
        xticklabels=True,
        yticklabels=True,

        # Clustering
        method="average",
        metric="euclidean",

        # Appearance
        linewidths=0,
        dendrogram_ratio=(0.12, 0.02),

        # Colorbar position:
        # separated from the row dendrogram
        cbar_pos=(0.02, 0.82, 0.025, 0.14),

        cbar_kws={
            "ticks": [-2, -1, 0, 1, 2]
        }
    )


    # --------------------------------------------------------
    # J. FORMAT HEATMAP LABELS
    # --------------------------------------------------------

    cluster.ax_heatmap.set_xlabel(
        "Condition and sex",
        fontsize=10,
        labelpad=8
    )

    cluster.ax_heatmap.set_ylabel("")


    cluster.ax_heatmap.tick_params(
        axis="x",
        labelsize=10,
        rotation=0
    )

    cluster.ax_heatmap.tick_params(
        axis="y",
        labelsize=ytick_fontsize,
        length=0
    )


    # Put Z-score label above the colorbar
    # instead of over the dendrogram.

    cluster.cax.set_ylabel("")

    cluster.cax.set_title(
        "Z-score",
        fontsize=8,
        pad=5
    )


    cluster.fig.suptitle(
        category_spec["title"],
        fontsize=14,
        fontweight="bold",
        y=1.02
    )


    # --------------------------------------------------------
    # K. SAVE HEATMAP
    # --------------------------------------------------------

    slug = category_spec["slug"]


    png_path = os.path.join(
        OUTPUT_DIR,
        slug + "_heatmap.png"
    )


    cluster.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight"
    )


    # Optional PDF version

    pdf_path = None

    if SAVE_PDF:

        pdf_path = os.path.join(
            OUTPUT_DIR,
            slug + "_heatmap.pdf"
        )

        cluster.savefig(
            pdf_path,
            bbox_inches="tight"
        )


    plt.close(cluster.fig)


    # --------------------------------------------------------
    # L. EXPORT PROTEIN MEMBERSHIP AND QUALITY INFORMATION
    # --------------------------------------------------------

    member_table = matched_annotation[
        [
            "protein_group_id",
            "gene_name",
            "matching_gene_symbols_text"
        ]
    ].copy()


    member_table["passed_coverage_filter"] = (
        member_table["protein_group_id"]
        .isin(means_valid.index)
    )


    member_table["included_in_heatmap"] = (
        member_table["protein_group_id"]
        .isin(z_scores.index)
    )


    # Add subgroup coverage, mean and Z-score

    for group in GROUP_ORDER:

        display_name = DISPLAY_GROUPS[group]


        member_table[
            f"coverage_{display_name}"
        ] = (
            member_table["protein_group_id"]
            .map(coverage[group])
        )


        member_table[
            f"mean_log2_{display_name}"
        ] = (
            member_table["protein_group_id"]
            .map(means[group])
        )


        member_table[
            f"zscore_{display_name}"
        ] = (
            member_table["protein_group_id"]
            .map(z_scores[group])
        )


    members_path = os.path.join(
        OUTPUT_DIR,
        slug + "_protein_members.csv"
    )


    member_table.to_csv(
        members_path,
        index=False
    )


    # --------------------------------------------------------
    # M. EXPORT ENRICHED TERMS
    # --------------------------------------------------------

    terms_path = os.path.join(
        OUTPUT_DIR,
        slug + "_enrichment_terms.csv"
    )


    selected_terms.to_csv(
        terms_path,
        index=False
    )


    # --------------------------------------------------------
    # N. EXPORT GENE LIST AND CURATION
    # --------------------------------------------------------

    gene_audit = pd.DataFrame({

        "gene_symbol": sorted(
            selected_genes_original
        )

    })


    gene_audit["selected_for_heatmap"] = (
        ~gene_audit["gene_symbol"].isin(excluded_genes)
    )


    gene_audit["manual_exclusion_reason"] = (
        gene_audit["gene_symbol"].map({

            "FGA": "Coagulation / fibrinogen",
            "FGB": "Coagulation / fibrinogen",
            "FGG": "Coagulation / fibrinogen",
            "PLG": "Fibrinolysis",
            "SERPINF2": "Fibrinolysis",
            "C1QC": "Complement",
            "ACTN1": "Cytoskeleton / adhesion",
            "PLEC": "Cytoskeleton / adhesion"

        })
    )


    gene_audit_path = os.path.join(
        OUTPUT_DIR,
        slug + "_gene_audit.csv"
    )


    gene_audit.to_csv(
        gene_audit_path,
        index=False
    )


    # --------------------------------------------------------
    # O. SUMMARY
    # --------------------------------------------------------

    result = {

        "category": category_spec["title"],

        "enriched_terms_found": len(selected_terms),

        "genes_before_curation": len(selected_genes_original),

        "genes_reclassified_out": len(excluded_genes),

        "genes_after_curation": len(selected_genes),

        "unmatched_gene_symbols": len(unmatched_genes),

        "protein_groups_matched": len(matched_ids),

        "protein_groups_passed_coverage": int(
            valid_coverage_mask.sum()
        ),

        "proteins_plotted": len(z_scores),

        "heatmap_png": png_path,

        "members_csv": members_path,

        "terms_csv": terms_path,

        "gene_audit_csv": gene_audit_path
    }


    if pdf_path is not None:

        result["heatmap_pdf"] = pdf_path


    print("\nCATEGORY SUMMARY")
    print("------------------------------")

    for key, value in result.items():

        print(f"{key}: {value}")


    return result


# ------------------------------------------------------------
# 17. GENERATE ALL FIVE HEATMAPS
# ------------------------------------------------------------

summary_results = []


for category_spec in CATEGORY_SPECS:

    result = create_category_heatmap(
        category_spec
    )

    if result is not None:

        summary_results.append(result)


# ------------------------------------------------------------
# 18. SAVE SUMMARY TABLE
# ------------------------------------------------------------

summary_df = pd.DataFrame(
    summary_results
)


summary_path = os.path.join(
    OUTPUT_DIR,
    "heatmap_summary.csv"
)


summary_df.to_csv(
    summary_path,
    index=False
)


print("\n")
print("=" * 70)
print("HEATMAP GENERATION COMPLETED")
print("=" * 70)


display(summary_df)


# ------------------------------------------------------------
# 19. CREATE ZIP FILE
# ------------------------------------------------------------

zip_filename = "PR1_five_heatmaps_outputs.zip"


with zipfile.ZipFile(
    zip_filename,
    "w",
    zipfile.ZIP_DEFLATED
) as zip_file:

    for filename in os.listdir(OUTPUT_DIR):

        full_path = os.path.join(
            OUTPUT_DIR,
            filename
        )

        zip_file.write(
            full_path,
            arcname=filename
        )


print("\nOutput folder:", OUTPUT_DIR)

print("ZIP file:", zip_filename)


# ------------------------------------------------------------
# 20. DOWNLOAD ALL RESULTS
# ------------------------------------------------------------

from google.colab import files as colab_files

colab_files.download(
    zip_filename
)

FILES LOADED
------------------------------
Expression: (3834, 76)
Protein annotation: (3834, 5)
Sample annotation: (75, 3)
gProfiler UP: (115, 7)
gProfiler DOWN: (11, 7)

Expression matrix: (3834, 75)

SAMPLE COUNTS
------------------------------
CON-F : 8
CON-M : 9
AS-F : 21
AS-M : 20

PEPTIDE SUPPORT FILTER
------------------------------
Protein groups excluded due to low peptide support: 79


ECM composition and remodeling
Enriched terms found: 9
Gene symbols before manual curation: 64
Genes reclassified out of this category: ['ACTN1', 'C1QC', 'FGA', 'FGB', 'FGG', 'PLEC', 'PLG', 'SERPINF2']
Gene symbols after curation: 56
Matched protein groups after peptide filtering: 61
Selected genes not matched to protein annotation: []
Protein groups passing coverage filter: 58

CATEGORY SUMMARY
------------------------------
category: ECM composition and remodeling
enriched_terms_found: 9
genes_before_curation: 64
genes_reclassified_out: 8
genes_after_curation: 56
unmatched_gene_symbols: 0
pr

,category,enriched_terms_found,genes_before_curation,genes_reclassified_out,genes_after_curation,unmatched_gene_symbols,protein_groups_matched,protein_groups_passed_coverage,proteins_plotted,heatmap_png,members_csv,terms_csv,gene_audit_csv
0,ECM composition and remodeling,9,64,8,56,0,61,58,58,heatmap_outputs/01_ECM_heatmap.png,heatmap_outputs/01_ECM_protein_members.csv,heatmap_outputs/01_ECM_enrichment_terms.csv,heatmap_outputs/01_ECM_gene_audit.csv
1,Energy metabolism and mitochondria,3,154,0,154,2,153,151,151,heatmap_outputs/02_Mitochondria_heatmap.png,heatmap_outputs/02_Mitochondria_protein_member...,heatmap_outputs/02_Mitochondria_enrichment_ter...,heatmap_outputs/02_Mitochondria_gene_audit.csv
2,Protein synthesis and ribosomal machinery,7,71,0,71,0,71,70,70,heatmap_outputs/03_Protein_synthesis_heatmap.png,heatmap_outputs/03_Protein_synthesis_protein_m...,heatmap_outputs/03_Protein_synthesis_enrichmen...,heatmap_outputs/03_Protein_synthesis_gene_audi...
3,"Cytoskeleton, adhesion and muscle contraction",5,78,0,78,0,99,95,95,heatmap_outputs/04_Cytoskeleton_heatmap.png,heatmap_outputs/04_Cytoskeleton_protein_member...,heatmap_outputs/04_Cytoskeleton_enrichment_ter...,heatmap_outputs/04_Cytoskeleton_gene_audit.csv
4,"Complement, coagulation and immune-associated ...",34,83,0,83,0,89,88,88,heatmap_outputs/05_Complement_coagulation_heat...,heatmap_outputs/05_Complement_coagulation_prot...,heatmap_outputs/05_Complement_coagulation_enri...,heatmap_outputs/05_Complement_coagulation_gene...



Output folder: heatmap_outputs
ZIP file: PR1_five_heatmaps_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>